# Splunk — First Contact

Splunk is an operational intelligence and observability platform built to ingest machine data, index it, and let engineers search it fast. Instead of thinking in rows and joins first, think in events. Logs, alerts, metrics-derived notifications, and security-relevant signals flow into Splunk as indexed events that can be searched almost immediately.

The core mental model is simple: machine data comes in, Splunk indexes the events, and engineers use SPL (Search Processing Language) to filter, aggregate, correlate, and visualize what happened. Splunk sits at the intersection of log aggregation, operational monitoring, and SIEM-style workflows. It is where ops and security teams ask questions like: what failed, where, how often, and is it getting worse right now?

In the Citi framing, Kafka handles stream ingestion. Splunk handles operational intelligence — ops teams write SPL searches to find endpoint anomalies, build dashboards, and trigger alerts in real time.

```text
[Postgres alerts] -> [Python HEC sender] -> [Splunk Indexer] -> [SPL Search] -> [Dashboard]
```

In [1]:
%pip install requests psycopg2-binary

Note: you may need to restart the kernel to use updated packages.


In [2]:
import requests
import psycopg2
import json
import time
from datetime import datetime, timezone


Verify Splunk is running before we ingest data.

In [3]:
requests.packages.urllib3.disable_warnings()

SPLUNK_BASE = "https://localhost:8089"
SPLUNK_AUTH = ("admin", "DeSplunk2026!")

try:
    resp = requests.get(
        f"{SPLUNK_BASE}/services/server/info",
        params={"output_mode": "json"},
        auth=SPLUNK_AUTH,
        verify=False,
        timeout=15,
    )
    resp.raise_for_status()
    payload = resp.json()
    entry = payload.get("entry", [{}])[0]
    content = entry.get("content", {})
    version = content.get("version", "unknown")
    build = content.get("build", "unknown")
    server_name = content.get("serverName", "unknown")
    print(f"Splunk version: {version}")
    print(f"Build: {build}")
    print(f"Server name: {server_name}")
except Exception as exc:
    print("citi_splunk container not running — check docker ps")
    print("Also confirm Splunk management API is reachable at https://localhost:8089 and credentials are admin / DeSplunk2026!.")
    raise

Splunk version: 9.2.0
Build: 1fff88043d5f
Server name: c79f6d39375a


Pull 500 alerts from Postgres — enrich with endpoint metadata.

In [4]:
PG_CONN = {
    "host": "localhost",
    "port": 5432,
    "dbname": "de_telemetry",
    "user": "de_admin",
    "password": "DeAdmin2026!",
}

query = """
SELECT
    a.alert_id,
    a.endpoint_id,
    a.severity,
    a.message,
    a.created_at,
    e.name AS endpoint_name,
    e.region,
    e.category
FROM alerts a
JOIN endpoints e USING (endpoint_id)
ORDER BY a.created_at DESC
LIMIT 500
"""

alerts = []
with psycopg2.connect(**PG_CONN) as conn:
    with conn.cursor() as cur:
        cur.execute(query)
        columns = [desc[0] for desc in cur.description]
        for row in cur.fetchall():
            record = dict(zip(columns, row))
            created_at = record["created_at"]
            if isinstance(created_at, datetime):
                if created_at.tzinfo is None:
                    created_at = created_at.replace(tzinfo=timezone.utc)
                record["created_at"] = created_at.isoformat()
            alerts.append(record)

print(f"Loaded {len(alerts)} enriched alerts from Postgres")

Loaded 500 enriched alerts from Postgres

## HEC — HTTP Event Collector

HEC is Splunk's high-throughput ingestion endpoint. It accepts token-authenticated HTTP requests, which makes it ideal for application-side or notebook-side data shipping. Each event is sent as a JSON envelope with metadata such as `time`, `host`, `source`, `sourcetype`, `index`, and the actual `event` payload.

In this notebook, we send the enriched alert records from Postgres into Splunk using HTTPS on port 8088. Because the local container uses a self-signed certificate, requests are sent with `verify=False`.

In [5]:
HEC_URL = "https://localhost:8088/services/collector/event"
HEC_TOKEN = "f9d0f92a-fcad-4a02-a76e-0b9a325cffe8"
headers = {
    "Authorization": f"Splunk {HEC_TOKEN}",
    "Content-Type": "application/json",
}

def to_unix_ts(value: str) -> float:
    dt = datetime.fromisoformat(value)
    if dt.tzinfo is None:
        dt = dt.replace(tzinfo=timezone.utc)
    return dt.timestamp()

def build_hec_event(alert: dict) -> dict:
    return {
        "time": to_unix_ts(alert["created_at"]),
        "host": "postgres-exporter",
        "source": "citi_telemetry",
        "sourcetype": "citi:alert",
        "index": "citi_telemetry",
        "event": alert,
    }

batch_size = 50
sent_total = 0

for start in range(0, len(alerts), batch_size):
    batch = alerts[start:start + batch_size]
    payload = "\n".join(json.dumps(build_hec_event(alert)) for alert in batch)
    try:
        resp = requests.post(
            HEC_URL,
            headers=headers,
            data=payload.encode("utf-8"),
            verify=False,
            timeout=30,
        )
        if resp.status_code != 200:
            print(f"HEC batch starting at {start} failed: {resp.status_code} {resp.text}")
            continue
        sent_total += len(batch)
        if sent_total % 100 == 0:
            print(f"Sent {sent_total} events so far...")
    except Exception as exc:
        print(f"HEC batch starting at {start} raised error: {exc}")
        continue

print(f"Sent {sent_total} events to Splunk HEC")

Sent 100 events so far...

Sent 200 events so far...


Sent 300 events so far...


Sent 400 events so far...


Sent 500 events so far...
Sent 500 events to Splunk HEC


In [6]:
print("Waiting 5s for Splunk to index events...")
time.sleep(5)

Waiting 5s for Splunk to index events...


## SPL — Search Processing Language

SPL is Splunk's query language. The basic pattern is pipeline-oriented: start with a search, then transform or enrich the result set, then output aggregated or formatted results. The base search for this notebook is:

`index=citi_telemetry sourcetype=citi:alert`

We will execute searches through the Splunk REST API so the notebook stays fully programmatic and repeatable.

In [7]:
def run_spl(query, count=20):
    search_url = f"{SPLUNK_BASE}/services/search/jobs/export"
    data = {
        "search": query,
        "output_mode": "json",
        "exec_mode": "blocking",
        "count": count,
    }
    try:
        resp = requests.post(
            search_url,
            data=data,
            auth=SPLUNK_AUTH,
            verify=False,
            timeout=60,
        )
        if resp.status_code != 200:
            print(f"Search failed for query: {query}")
            print(f"Status: {resp.status_code} | Body: {resp.text}")
            return []
        results = []
        for line in resp.text.splitlines():
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
            except json.JSONDecodeError:
                continue
            if "result" in obj:
                results.append(obj["result"])
            elif "messages" in obj:
                for msg in obj["messages"]:
                    text = msg.get("text")
                    if text:
                        print(f"Splunk message: {text}")
        return results
    except Exception as exc:
        print(f"Search error for query: {query}")
        print(exc)
        return []

def print_results(title, rows):
    print(f"\n{title}")
    print("-" * len(title))
    if not rows:
        print("No results returned")
        return
    for idx, row in enumerate(rows, start=1):
        print(f"{idx}. {json.dumps(row, indent=2)}")

# NOTE: HEC stores the event dict directly as _raw (flat JSON).
# Fields like region, endpoint_name, severity are at the top level — use spath <field>, not spath event.<field>
q1 = 'search index=citi_telemetry sourcetype=citi:alert | spath severity | stats count by severity | sort -count'
q2 = 'search index=citi_telemetry sourcetype=citi:alert | spath region | stats count by region | sort -count'
q3 = 'search index=citi_telemetry sourcetype=citi:alert | spath endpoint_name | top limit=10 endpoint_name'
q4 = 'search index=citi_telemetry sourcetype=citi:alert | spath severity | search severity=CRITICAL | stats count'

r1 = run_spl(q1, count=20)
print_results("Query 1 — Count by severity", r1)

r2 = run_spl(q2, count=20)
print_results("Query 2 — Count by region", r2)

r3 = run_spl(q3, count=10)
print_results("Query 3 — Top 10 endpoints by alert count", r3)

r4 = run_spl(q4, count=20)
print_results("Query 4 — CRITICAL alerts in index", r4)


Query 1 — Count by severity
---------------------------
1. {
  "severity": "MEDIUM",
  "count": "288"
}
2. {
  "severity": "LOW",
  "count": "262"
}
3. {
  "severity": "HIGH",
  "count": "227"
}
4. {
  "severity": "CRITICAL",
  "count": "224"
}
5. {
  "severity": "medium",
  "count": "144"
}
6. {
  "severity": "low",
  "count": "131"
}
7. {
  "severity": "high",
  "count": "113"
}
8. {
  "severity": "critical",
  "count": "112"
}

Query 2 — Count by region
-------------------------
1. {
  "region": "NYC1",
  "count": "393"
}
2. {
  "region": "LON1",
  "count": "384"
}
3. {
  "region": "SNG1",
  "count": "366"
}
4. {
  "region": "NYC2",
  "count": "357"
}



Query 3 — Top 10 endpoints by alert count
-----------------------------------------
1. {
  "endpoint_name": "srv-07813.citi.internal",
  "count": "6",
  "percent": "0.400000",
  "_tc": "1500"
}
2. {
  "endpoint_name": "srv-07004.citi.internal",
  "count": "6",
  "percent": "0.400000",
  "_tc": "1500"
}
3. {
  "endpoint_name": "srv-06741.citi.internal",
  "count": "6",
  "percent": "0.400000",
  "_tc": "1500"
}
4. {
  "endpoint_name": "srv-03407.citi.internal",
  "count": "6",
  "percent": "0.400000",
  "_tc": "1500"
}
5. {
  "endpoint_name": "srv-02478.citi.internal",
  "count": "6",
  "percent": "0.400000",
  "_tc": "1500"
}
6. {
  "endpoint_name": "srv-01677.citi.internal",
  "count": "6",
  "percent": "0.400000",
  "_tc": "1500"
}
7. {
  "endpoint_name": "srv-09997.citi.internal",
  "count": "3",
  "percent": "0.200000",
  "_tc": "1500"
}
8. {
  "endpoint_name": "srv-09916.citi.internal",
  "count": "3",
  "percent": "0.200000",
  "_tc": "1500"
}
9. {
  "endpoint_name": "srv-09910.

## SPL — Commands You Must Know

| Command | What it does | Example |
|---------|-------------|---------|
| search | Filter events (implicit at start) | search severity=CRITICAL |
| stats | Aggregate (count, avg, sum, dc) | stats count by region |
| timechart | Time-series aggregation | timechart span=1h count by severity |
| eval | Compute new fields | eval is_critical=if(severity=\"CRITICAL\",1,0) |
| rex | Extract fields with regex | rex field=message \"endpoint=(?<ep>\\w+)\" |
| top/rare | Most/least common values | top 10 endpoint_name |
| dedup | Remove duplicate events | dedup endpoint_id |
| sort | Order results | sort -count |
| spath | Extract from nested JSON | spath event.region |
| transaction | Group events into sessions | transaction endpoint_id maxspan=1h |

## Splunk UI — Where to Find Things

- http://localhost:8000 — Splunk Web (admin/DeSplunk2026!)
- Apps → Search & Reporting → Search bar → paste SPL queries
- Save As → Dashboard Panel to build dashboards
- Settings → Indexes → verify `citi_telemetry` index exists
- Settings → Data Inputs → HTTP Event Collector → verify `citi_telemetry_hec` is enabled
- Data Summary → Sourcetypes → `citi:alert` → view your events

## What Just Happened

- Built the Splunk mental model: machine data becomes indexed events, then searchable operational intelligence.
- Used HEC to ingest enriched telemetry alerts from Postgres into Splunk.
- Indexed 500 enriched events into the `citi_telemetry` index.
- Ran 4 SPL queries through the REST API.
- Used the REST API as a programmatic path for search and automation.

Citi tie-in: An ops engineer at Citi types:

`index=citi_telemetry sourcetype=citi:alert severity=CRITICAL | timechart span=15m count`

— real-time view of critical alerts per 15-minute window, no SQL, no Python.

Next: Run `splunk_concepts.md` for vocabulary, then Round 2 for SPL internals and Citi narrative.